In [1]:
pip show tensorflow

Name: tensorflow
Version: 2.21.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: c:\Users\rusir\AppData\Local\Programs\Python\Python311\Lib\site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google_pasta, grpcio, h5py, keras, libclang, ml_dtypes, numpy, opt_einsum, packaging, protobuf, requests, setuptools, six, termcolor, typing_extensions, wrapt
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout

from sklearn.metrics import accuracy_score, classification_report

In [3]:
df = pd.read_csv("../data/cleaned/spam.csv")

# Remove empty rows
df = df.dropna(subset=['clean_message'])
df = df[df['clean_message'].str.strip() != ""]

df.head()

,label,message,clean_message
0,ham,"Go until jurong point, crazy.. Available only ...",go jurong point crazy available bugis n great ...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry wkly comp win fa cup final tkts st ...
3,ham,U dun say so early hor... U c already then say...,u dun say early hor u c already say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah dont think go usf life around though


In [4]:
encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['label'])

In [5]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(df['clean_message'])

X = tokenizer.texts_to_sequences(df['clean_message'])

In [6]:
X = pad_sequences(X, maxlen=100)

y = df['label']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(4452, 100)
(1114, 100)


In [8]:
print(X_train.shape)
print(X_test.shape)

(4452, 100)
(1114, 100)


In [9]:
lstm_model = Sequential()

lstm_model.add(Embedding(input_dim=5000, output_dim=64, input_length=100))

lstm_model.add(LSTM(64))

lstm_model.add(Dropout(0.5))

lstm_model.add(Dense(1, activation='sigmoid'))

lstm_model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

lstm_model.summary()

c:\Users\rusir\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
history = lstm_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.9059 - loss: 0.2734 - val_accuracy: 0.9697 - val_loss: 0.1250
Epoch 2/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.9834 - loss: 0.0662 - val_accuracy: 0.9787 - val_loss: 0.0734
Epoch 3/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.9919 - loss: 0.0315 - val_accuracy: 0.9820 - val_loss: 0.0721
Epoch 4/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 24ms/step - accuracy: 0.9966 - loss: 0.0151 - val_accuracy: 0.9820 - val_loss: 0.0690
Epoch 5/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 0.9978 - loss: 0.0098 - val_accuracy: 0.9832 - val_loss: 0.0780


In [11]:
loss, accuracy = lstm_model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9856 - loss: 0.0537
Test Accuracy: 0.985637366771698


In [12]:
y_pred = (lstm_model.predict(X_test) > 0.5).astype("int32")

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step


In [13]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       979
           1       0.95      0.93      0.94       135

    accuracy                           0.99      1114
   macro avg       0.97      0.96      0.97      1114
weighted avg       0.99      0.99      0.99      1114



In [14]:
loss, accuracy = lstm_model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9856 - loss: 0.0537
Test Accuracy: 0.985637366771698


In [15]:
y_pred = (lstm_model.predict(X_test) > 0.5).astype(int)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step


In [16]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       979
           1       0.95      0.93      0.94       135

    accuracy                           0.99      1114
   macro avg       0.97      0.96      0.97      1114
weighted avg       0.99      0.99      0.99      1114

